#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col, regexp_replace, to_date
from pyspark.sql.window import Window
import uuid
from datetime import datetime

In [0]:
# erp_cust_az12_raw time (starts here...)

start_time = datetime.now()

# Reading From Bronze

In [0]:
df = spark.table("salesdatalakehouse.bronze.erp_cust_az12_raw")

# Data Transformations

## Remove Invalid char from Customer ID

In [0]:
df = df.withColumn("cid", regexp_replace(col("cid"), "NAS", ""))

##Handle Dates

In [0]:
# when birth date  is less than 1926-01-01 or greater than current date, replace with null

df = (
    df
        .withColumn(
            "BDATE",
            F.when((col("BDATE") < "1926-01-01") | (col("BDATE") > F.current_date()), None)
            .otherwise(col("BDATE"))
        )
    )

## Normalization

In [0]:
df = (
    df
    .withColumn("GEN",
                F.when(F.upper(trim(col("GEN"))).isin("F", 'FEMALE'), "Female")
                .when(F.upper(trim(col("GEN"))).isin("M", 'MALE'), "Male")
                .otherwise('n/a')
    )
)

## Renaming Colmns

In [0]:
df = df.toDF(
    "customer_id",
    "birth_date",
    "gender"
)

# Load Into Silver

In [0]:
df.write.format("delta").mode("overwrite").option('overwriteSchema', 'true').saveAsTable('salesdatalakehouse.silver.erp_cust_az12')

#erp_cust_az12 Logging

In [0]:

# erp_cust_az12_raw time (ends here...)

run_id = str(uuid.uuid4())
job_name = "sales_etl"
task_name = 'load_Silver'
schema_name = 'silver'
table_name = 'erp_cust_az12'
row_inserted = spark.sql(''' SELECT COUNT(*) FROM salesdatalakehouse.silver.erp_cust_az12''').collect()[0][0]
end_time = datetime.now()

# Logging into pipeline_runs table
spark.sql(f"""
  INSERT INTO salesdatalakehouse.audit.pipeline_runs
  VALUES (
    '{run_id}',
    '{job_name}',
    '{task_name}',
    '{schema_name}',
    '{table_name}',
    '{start_time}',
    '{end_time}',
    DATEDIFF(SECOND, '{start_time}', '{end_time}'),
    'SUCCESS',
    '{row_inserted}',
    0,
    0,
    NULL,
    CURRENT_TIMESTAMP()
  )
""")

print(f"✓ erp_cust_az12 info Logged with run {run_id}")